# 08 Detection baseline (M2-1): 20-image timing sample
Runs on Colab, **fresh runtime**, CPU. Plan: PLAN.md (M2, decisions of 2026-11-15).

**This notebook only measures cost.** It loads baseline A (Babblu2821/alpr-plate-detector) before Drive is mounted, records its MD5, then times 20 images per test set. `model.val` runs on the 20 images only to time it; its metrics are **not printed** on purpose (no score before the full run). No plate images or text are shown.

In [ ]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition
uv pip install --system -q ultralytics huggingface_hub

In [ ]:
import os, platform, random, shutil, statistics, tarfile, time
from pathlib import Path
import torch
import ultralytics
from google.colab import drive
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from ultralytics import YOLO
from alpr import ccpd

In [ ]:
# Parameters
SEED = 0
REPO_A, WEIGHTS_A = 'Babblu2821/alpr-plate-detector', 'best.pt'   # candidate A (PLAN.md)
N_SAMPLE, REPEATS = 20, 3          # images per set, timing passes per image
IMGSZ, CONF, IOU = 640, 0.001, 0.6  # same for every detector
DRIVE = '/content/drive/MyDrive/alpr'
LOCAL, WORK = '/content/tars', '/content/work'
SETS = {   # name -> (tar, MD5 recorded in PLAN.md, split folder, images in the full set)
    'us': ('us_yolo.tar', '1ce491ae6a7accefdb0b9156a30f2acc', 'test', 105),
    'ccpd': ('ccpd_yolo.tar', 'bd44986acd6691a4f90fbea1e6017682', 'heldout', 300),
    'uc3m_395': ('uc3m_yolo.tar', 'e3ef306731f33095bac983338c6adb04', 'test', 395),
}
UC3M_ALL = 1975

In [ ]:
cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), platform.processor())
print('CPU:', cpu, '| cores:', os.cpu_count(), '| python', platform.python_version(), '| ultralytics', ultralytics.__version__, '| torch', torch.__version__)
print('GPU visible:', torch.cuda.is_available(), '(all runs below use device=cpu)')

In [ ]:
# 1. Weights BEFORE Drive is mounted: download to the VM disk, record the MD5, load the model.
assert not os.path.ismount('/content/drive'), 'restart the runtime: Drive must not be mounted yet'
info = HfApi().model_info(REPO_A)
print('HF commit:', info.sha, '| licence tag:', info.card_data.license if info.card_data else None)
wpath = hf_hub_download(REPO_A, WEIGHTS_A, revision=info.sha, local_dir=f'{WORK}/weights')
W_MD5 = ccpd.file_md5(wpath)
print(f'{WEIGHTS_A}: {Path(wpath).stat().st_size / 1e6:.2f} MB, MD5 {W_MD5}')
model = YOLO(wpath)
assert len(model.names) == 1, model.names   # single class: val matches class 0 of our labels
print('class names:', model.names, '| task:', model.task)

In [ ]:
# 2. Only now mount Drive and copy the weights there (MD5 checked).
drive.mount('/content/drive')
dst = Path(DRIVE, 'weights', 'baseline_A_best.pt')
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(wpath, dst)
assert ccpd.file_md5(dst) == W_MD5, 'Drive copy differs'
print('copied to', dst, '| record in PLAN.md: HF commit', info.sha, 'MD5', W_MD5)

In [ ]:
# 3. Test tars to the VM disk, MD5 against PLAN.md (time reported: part of the full-run cost).
Path(LOCAL).mkdir(exist_ok=True)
for name, (fn, md5, _, _) in SETS.items():
    src, d = Path(DRIVE, 'archives', fn), Path(LOCAL, fn)
    assert src.exists(), f'{fn} not found'
    t0 = time.perf_counter()
    if not d.exists():
        shutil.copyfile(src, d)
    t1 = time.perf_counter()
    got = ccpd.file_md5(d)
    assert got == md5, f'{name}: MD5 {got} differs from PLAN.md'
    print(f'{name}: copy {t1 - t0:.0f} s, MD5 check {time.perf_counter() - t1:.0f} s, ok')

In [ ]:
# 4. Extract a seeded sample of N_SAMPLE images + labels per set, and write a data yaml for each.
SAMPLE = {}
for name, (fn, _, split, _) in SETS.items():
    root = Path(WORK, 'sample', name)
    (root / 'images').mkdir(parents=True, exist_ok=True)
    (root / 'labels').mkdir(exist_ok=True)
    with tarfile.open(Path(LOCAL, fn)) as t:
        names = sorted(m.name for m in t.getmembers() if m.isfile() and m.name.startswith(f'images/{split}/'))
        for n in random.Random(SEED).sample(names, N_SAMPLE):
            stem = Path(n).stem
            lab = f'labels/{split}/{stem}.txt'
            (root / 'images' / Path(n).name).write_bytes(t.extractfile(n).read())
            (root / 'labels' / f'{stem}.txt').write_bytes(t.extractfile(lab).read())
    (root / 'data.yaml').write_text(f'path: {root}\nval: images\nnames:\n  0: plate\n')
    SAMPLE[name] = sorted((root / 'images').iterdir())
    print(name, len(SAMPLE[name]), 'images')

In [ ]:
# 5. Timing: decode and predict per image (warm-up excluded), then wall time of model.val on the sample (metrics discarded, not printed).
def spread(v):
    q = statistics.quantiles(v, n=4)
    return f'median {statistics.median(v) * 1000:.0f} ms | min {min(v) * 1000:.0f} | IQR {q[0] * 1000:.0f}-{q[2] * 1000:.0f} | max {max(v) * 1000:.0f}'

full = {'us': 105, 'ccpd': 300, 'uc3m_395': 395}
model.predict(str(SAMPLE['us'][0]), imgsz=IMGSZ, device='cpu', verbose=False)   # warm-up
for name, files in SAMPLE.items():
    dec, pred = [], []
    for f in files:
        t0 = time.perf_counter()
        with Image.open(f) as im:
            im.load()
        dec.append(time.perf_counter() - t0)
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            model.predict(str(f), imgsz=IMGSZ, conf=CONF, iou=IOU, device='cpu', verbose=False)
            pred.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    r = model.val(data=str(SAMPLE[name][0].parents[1] / 'data.yaml'), imgsz=IMGSZ, conf=CONF, iou=IOU, device='cpu', batch=1,
                  workers=2, plots=False, verbose=False, seed=SEED, project=f'{WORK}/runs', name='sample', exist_ok=True)
    val_s = time.perf_counter() - t0
    n = full[name]
    print(f'{name}: decode {spread(dec)}\n    predict {spread(pred)}\n    val of {N_SAMPLE}: {val_s:.0f} s -> full {n} images ~ {val_s / N_SAMPLE * n / 60:.1f} min at imgsz {IMGSZ}')
print(f'uc3m all {UC3M_ALL} images: scale the uc3m_395 line by {UC3M_ALL / 395:.1f}')